# Lab 1: Building an Instruction-Tuning Dataset

**Time:** about 75 minutes

Fine-tuning teaches a model by example. Every example is a small conversation: an instruction or question, and the answer
you want. The model will copy what it sees, **including the mistakes**. An outdated answer in the training data becomes
an outdated answer the model gives confidently, forever, to every member who asks.

So this lab is about data, not models. You'll take the Fund's website FAQ export and turn it into a dataset you'd be
willing to sign off:

```
faq_export.csv ─► clean HTML ─► remove drafts, blanks, archived ─► fix answers superseded by circulars
              ─► group paraphrases ─► add behaviour examples (refusals, redirections) ─► chat format
              ─► split without leakage ─► check against held-out questions ─► PII scan ─► data card
```

### What is instruction tuning?

On Day 7 you used **instruction-tuned** models: models that had extra training on examples of instructions and good
answers, so that they behave like helpful assistants. **Instruction tuning** is that extra training. Today and in Lab 2
you'll do a small version of it yourself: show a model a few hundred examples of member questions and the Fund's
correct answers, so it learns to answer in the Fund's way.

It's like training a new call-centre agent with a binder of past questions and model answers. If the binder contains an
answer from 2023 that a 2025 circular changed, the new agent will learn the wrong rule, and say it with confidence. The
binder is the most important part of the training.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **fine-tuning** | extra training of a pre-trained model on your own examples |
| **instruction tuning** | fine-tuning on examples of instructions (questions) and the answers you want |
| **training example** | one question with its answer, written as a short conversation |
| **FAQ export** | the website's frequently-asked-questions, downloaded from the content system |
| **HTML** | the markup language of web pages: tags like `<p>` and `<br>` |
| **superseded** | replaced by a newer rule (here, by a circular) |
| **paraphrase** | the same question asked in different words |
| **leakage** | when the test set contains things the model saw in training, so the test measures memory |
| **behaviour example** | a training example that shows how to refuse or redirect, not a fact |
| **held-out set** | questions kept aside, never trained on, used for the final test |
| **cosine similarity** | a 0-to-1 measure of how similar two texts are (1 = same words) |
| **PII** | personally identifiable information: names, member numbers, phone numbers ... |
| **JSON Lines** | a file format with one JSON record per line |
| **data card** | a short document describing a dataset: sources, changes, limits, owner |

## Where this lab fits

**What you've done so far.** On Day 7 you ran open LLMs locally, made them useful with prompts, and chose a shortlist
with a scorecard. You saw that small models answering from memory get the Fund's policy wrong. On Day 6 you fine-tuned a
transformer to route complaints, and on Day 1 you cleaned raw data with rules you could audit.

**What you'll do now.** You'll prepare the data for fine-tuning an LLM: turn a messy website FAQ export into a clean set
of training conversations. You'll strip HTML, drop drafts and archived answers, correct published answers that circulars
have overtaken, group paraphrases, add examples of how to refuse and redirect, put everything in chat format, split it
without leakage, check it against the held-out questions, scan for personal data, and write a data card.

**Why it matters.** A fine-tuned model repeats what it was taught, with confidence. Every wrong, outdated or leaked
answer in the training data becomes a wrong, outdated or leaked answer given to members. Data preparation is most of
the work of fine-tuning and most of its value.

**What happens next.** Lab 2 fine-tunes a small model on the dataset you save here. Lab 3 tests the result against the
base model on the held-out questions.

### Theory: what fine-tuning can and can't teach

Fine-tuning changes the model's weights so that, given inputs like your examples, outputs like your answers become more
likely. In practice it's much better at some things than others:

| Fine-tuning teaches well | Fine-tuning teaches poorly |
|---|---|
| **style and tone**: brief, polite, the Fund's voice | **many new facts**: a few hundred examples can't reliably store a whole handbook |
| **format**: always one short paragraph, always cite the circular | **facts that change**: every rule change means retraining |
| **behaviour**: refuse what you can't see, redirect fraud attempts | **knowing what it doesn't know**: it may answer confidently anyway |
| **vocabulary**: Fund terms, Swahili phrasing | **reasoning it couldn't already do** |

Keep this table in mind for the whole day. It explains why Lab 3's results look the way they do, and why Day 9 gives the
model the right documents at question time instead of relying on what it memorised.

## 0. Set up today's environment (run this cell first)

You don't need to create a folder or an environment by hand today. The cell below does it for you:

1. It finds the environment you created on Day 1 (`DAY1/Labs/work/fund-ai/.venv`). If it isn't there, it creates it.
2. It installs today's libraries into it from ``requirements.txt``. The first time takes a few minutes; after that, seconds.
3. If this machine has an NVIDIA GPU (graphics processing unit), it switches PyTorch to its GPU build, so training runs much faster.
4. It connects the environment to Jupyter as **Python (fund-ai)**, and tells you if this notebook isn't using it yet.

It's safe to run again at any time. If it prints instructions to switch the kernel, follow them and run it once more.

You don't need to read the code in this cell. It's housekeeping, not part of today's lesson.

In [ ]:
# ---- Setup for Day 8. Safe to run more than once. -------------------------------------------------------------
COURSE_ENV = None        # None = use the Day 1 environment. Or set a folder, e.g. Path.home() / "fund-ai-venv"
REQUIREMENTS = "requirements.txt"
EXTRAS = ['ollama:gemma2:2b']

import os, shutil, subprocess, sys
from pathlib import Path

labs = Path.cwd()
root = next((p for p in [labs, *labs.parents] if (p / "DAY1").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the course folder (the one that contains DAY1, DAY2, ...).")
env = Path(COURSE_ENV) if COURSE_ENV else root / "DAY1" / "Labs" / "work" / "fund-ai" / ".venv"
py = env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def run(*cmd, quiet=False):
    """Run a command; show its output only if it fails."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0 and not quiet:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if "path" in (r.stderr + r.stdout).lower() and os.name == "nt":
            print("\nIf the error mentions a path that is too long: Windows 'long paths' are switched off on this machine.\n"
                  "Ask ICT to enable them, or set COURSE_ENV above to a short folder, e.g. Path.home() / 'fund-ai-venv'.")
        raise RuntimeError("A setup step failed. The details are printed above.")
    return r


# 1. The course environment
if py.exists():
    print(f"- course environment found: {env}")
else:
    print(f"- creating the course environment (first time only): {env}")
    run(getattr(sys, "_base_executable", sys.executable), "-m", "venv", env)
    run(py, "-m", "pip", "install", "-q", "--upgrade", "pip")

# 2. Today's libraries
req = labs / REQUIREMENTS
print(f"- installing today's libraries from {REQUIREMENTS} ...")
run(py, "-m", "pip", "install", "-q", "-r", req)

# 3. Use the GPU if this machine has an NVIDIA one
if "torch" in req.read_text() and shutil.which("nvidia-smi"):
    if run(py, "-c", "import torch; print(torch.cuda.is_available())").stdout.strip() != "True":
        version = run(py, "-c", "import torch; print(torch.__version__.split('+')[0])").stdout.strip()
        print(f"- NVIDIA GPU found: switching PyTorch {version} to its GPU build (about 2.5 GB, first time only) ...")
        run(py, "-m", "pip", "uninstall", "-y", "-q", "torch")
        run(py, "-m", "pip", "install", "-q", f"torch=={version}", "--index-url", "https://download.pytorch.org/whl/cu124",
            "--extra-index-url", "https://pypi.org/simple")
    print("- PyTorch will use the GPU")

# 4. Anything else today needs
for extra in EXTRAS:
    kind, name = extra.split(":", 1)
    if kind == "spacy" and run(py, "-c", f"import {name}", quiet=True).returncode != 0:
        print(f"- downloading the spaCy model {name} ...")
        run(py, "-m", "spacy", "download", name)
    if kind == "ollama":
        import json, urllib.request
        try:
            tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3))["models"]
            have = {m["name"] for m in tags}
            wanted = [m for m in name.split(",") if m]
            missing = [m for m in wanted if m not in have]
            print(f"- Ollama is running with {len(have)} model(s)" + (f"; still to pull: {', '.join(missing)}" if missing else ""))
            for m in missing:
                print(f"    in a terminal: ollama pull {m}")
        except OSError:
            print("- Ollama isn't running. Start the Ollama app (or run `ollama serve` in a terminal) before the LLM parts.")

# 5. Connect the environment to Jupyter
run(py, "-m", "ipykernel", "install", "--user", "--name", "fund-ai", "--display-name", "Python (fund-ai)")

# 6. Is this notebook using it?
if Path(sys.prefix).resolve() == env.resolve():
    print("\nReady. This notebook is running in the course environment. Carry on with the next cell.")
else:
    print("\nAlmost there. This notebook is running a different Python:")
    print(f"    {sys.executable}")
    print("Switch it to the course environment, then run this cell once more:")
    print("    1. Click the kernel name in the top right of the notebook, then 'Select Another Kernel'.")
    print("    2. Choose 'Jupyter Kernel' -> 'Python (fund-ai)'.")
    print("       (Or press Ctrl+Shift+P, run 'Python: Select Interpreter', choose 'Enter interpreter path...'")
    print("        and paste the path below, the same way you did on Day 1.)")
    print(f"    {py}")

## Imports

The tools for today's data work:

- `html`: turns HTML codes such as `&amp;` back into normal characters;
- `re`: regular expressions, for cleaning text and spotting patterns;
- `TfidfVectorizer` and `cosine_similarity`: to measure how similar two questions are (Section 8);
- `GroupShuffleSplit`: the train/validation split that keeps groups together, from Day 5;
- `requests`: to ask Ollama for draft examples (optional, Section 6).

`OUT` is the `data` folder where the finished dataset will be saved; `mkdir(exist_ok=True)` creates it if it isn't
there yet.

In [ ]:
import html                     # turns HTML codes like &amp; back into normal characters
import json
import re
from pathlib import Path

import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 170)
SEED = 42

# The finished dataset goes in a "data" folder.
OUT = Path("data")
OUT.mkdir(exist_ok=True)

> **Where we are.** The libraries are loaded and the `data` folder is ready for the finished dataset. First, find out
> what the export really contains.

## 1. What the export actually contains

`faq_export.csv` is a download from the website's content management system: one row per FAQ entry, with the question,
the answer as HTML (as it's stored for the web page), a `status` (published, draft, archived), a language and a
last-updated date. Count the statuses and languages first.

In [ ]:
faq = pd.read_csv("faq_export.csv")
print(faq.shape)
print(faq["status"].value_counts())
print()
print(faq["language"].value_counts())

Then look at a few rows. `sample(6, random_state=3)` picks the same 6 random rows each time.

In [ ]:
faq.sample(6, random_state=3)

Look through a dozen rows before writing any code. You'll find the usual state of a content-management export:

- HTML markup (`<p>`, `<span style=...>`, `&nbsp;`) and leftover labels like "Answer:",
- drafts, test entries and rows with an empty answer,
- **archived** answers that were replaced, and at least one **published** answer that's out of date,
- the same question asked several ways, sometimes with trailing punctuation typed twice,
- a handful of Swahili questions.

> **Where we are.** You've seen the usual state of a content-management export: HTML, drafts, empty and archived rows,
> outdated published answers, paraphrases and some Swahili. The work goes in the same order as Day 1's pipeline: clean,
> then remove, then correct.

## 2. Clean the text

A model trained on `<p>The pension is <b>40%</b>...` would learn to write HTML tags in its answers. So strip the markup
and keep only the words. Clean one answer step by step first, so you can see what each step does. `repr` prints the
text with its hidden characters visible (`\n` for a new line, `\xa0` for a non-breaking space).

The regular expressions used:

| Pattern | Means |
|---|---|
| `<br\s*/?>` | a line-break tag, written `<br>`, `<br/>` or `<br />` |
| `<[^>]+>` | `<`, then one or more characters that aren't `>`, then `>`: any HTML tag |
| `^\s*Answer:\s*` | "Answer:" at the very start, with any spaces around it |
| `[ \t]+` | one or more spaces or tabs |

In [ ]:
example = faq["answer_html"].iloc[0]
print("raw:     ", repr(example))

# 1. HTML codes to characters: &nbsp; -> a non-breaking space, &amp; -> &
step = html.unescape(example)

# 2. Line breaks (<br> and the end of a paragraph </p>) become real new lines.
step = re.sub(r"<br\s*/?>|</p>", "\n", step)

# 3. Remove every other tag: < followed by anything except > , then >.
step = re.sub(r"<[^>]+>", "", step)

# 4. Remove a leftover "Answer:" label at the start.
step = re.sub(r"^\s*Answer:\s*", "", step)

# 5. Tidy the spaces: non-breaking spaces to normal ones, repeated spaces to one, no spaces around new lines.
step = step.replace("\xa0", " ")
step = re.sub(r"[ \t]+", " ", step)
step = re.sub(r"\s*\n\s*", "\n", step).strip()

print("cleaned: ", repr(step))

The same steps as a function, so they can be applied to every answer, plus a second function that tidies each
question's punctuation. `isinstance(text, str)` checks that the answer really is text: an empty cell in a CSV comes in
as a missing value, not as text, and becomes `""` here.

`clean_question` removes a space before `?`, turns `??` into `?`, and adds a `?` to the end of a question that lacks one
(unless it's very short, under 3 words, which is probably not a real question). `rstrip(".")` removes a full stop from
the end first.

`map` applies a function to every value in a column.

In [ ]:
def clean_html(text):
    """The five cleaning steps above, for any answer. Empty answers become ""."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>|</p>", "\n", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"^\s*Answer:\s*", "", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text).strip()
    return text


def clean_question(question):
    """Tidy a question's punctuation: no space before '?', no '??', and a '?' at the end."""
    question = str(question).strip()
    question = re.sub(r"\s+\?", "?", question)        # "pension ?" -> "pension?"
    question = re.sub(r"\?{2,}", "?", question)       # "??" -> "?"
    if question.endswith("?") or len(question.split()) < 3:
        return question
    return question.rstrip(".") + "?"


faq["answer"] = faq["answer_html"].map(clean_html)
faq["question"] = faq["question"].map(clean_question)
faq[["answer_html", "answer"]].head(6)          # before and after, side by side

> **Where we are.** Answers are plain text without HTML or leftover labels, and questions have tidy punctuation. Some
> rows still shouldn't be taught at all.

## 3. Remove what should never be taught

Three kinds of row go. Make each one a True/False column, look at a few examples of each, then drop them all:

- **drafts and test entries:** never meant for members. `|` means "or"; `str.contains(..., case=False)` finds the
  text whatever its capitalisation;
- **empty answers:** fewer than 5 characters after cleaning;
- **archived answers:** replaced by a newer version, so out of date by definition.

In [ ]:
is_draft_or_test = (faq["status"] == "draft") | faq["question"].str.contains("test question", case=False)
is_empty = faq["answer"].str.len() < 5
is_archived = faq["status"] == "archived"

print("draft or test entry:", is_draft_or_test.sum(), "rows")
print("empty answer:       ", is_empty.sum(), "rows")
print("archived (replaced):", is_archived.sum(), "rows")

Always look at what you're about to delete, before you delete it:

In [ ]:
# Look at the archived rows before you remove them.
faq.loc[is_archived, ["faq_id", "question", "answer", "status", "last_updated"]]

Now remove all three kinds at once. `~` means "not", so `faq[~remove]` keeps the rows that are none of them.

In [ ]:
# Keep only the rows that are none of the three.
remove = is_draft_or_test | is_empty | is_archived
faq = faq[~remove].copy()
print("rows left:", len(faq))

Read the archived rows before you forget them. The archived maternity answer says 10 weeks; the handbook says 12.
The archived life-verification answer says it can only be done at a branch; circular FC/03/2025 added the app. That's
why "archived" means "never train on it".

> **Where we are.** Drafts, test entries, empty answers and archived answers are gone. What's left is officially
> "published". That doesn't make it right.

### Theory: the source of truth

Every dataset needs a **source of truth**: the authority that decides what's correct when two sources disagree. For the
Fund's rules, it's the handbook as amended by circulars, the newest circular winning. The website FAQ is only a copy,
and copies drift out of date.

A content system's `status` field records what someone *did* (published, archived), not whether the content is
*correct*. So checking against the source of truth has to be done separately, by someone who knows which rules changed.
Writing each known change as a rule (old wording, circular, corrected answer) turns that knowledge into a check that can
run on every future export, the same idea as Day 1's validation rules.

## 4. Published, but wrong: answers superseded by circulars

This is the dangerous category. The CMS (content management system) says these answers are live, but a later circular
changed the rule. Nobody updated the website. A model trained on them would repeat the old rule.

You can't find these automatically in general. It takes someone who knows which rules changed. What you *can* do is write
down every known change as a check, and run it on every future export. Each entry in `SUPERSEDED` has:

- `pattern`: a regular expression that finds the old wording;
- `circular`: the circular that changed the rule;
- `reason`: what changed, in plain words;
- `corrected`: the correct answer to use instead.

In [ ]:
SUPERSEDED = [
    {
        # (?<![\d.]) = "not preceded by a digit or a dot", so that "2.5%" doesn't match.
        "pattern": r"(?<![\d.])5% of the unpaid",
        "circular": "FC/09/2025",
        "reason": "Penalty reduced to 2.5% per month, capped at 50%, from 1 October 2025",
        "corrected": ("From 1 October 2025 the penalty is 2.5% of the unpaid contribution for each month or part of a month of "
                      "delay, capped at 50% of the unpaid contribution (circular FC/09/2025). Penalties are charged to the employer "
                      "and may not be deducted from employees' wages."),
    },
    {
        "pattern": r"amnesty|waived",
        "circular": "FC/09/2025",
        "reason": "The 2024 penalty amnesty has ended and was not renewed",
        "corrected": ("No. The one-time amnesty under circular FC/07/2024 ended on 31 December 2024 and has not been renewed "
                      "(circular FC/09/2025)."),
    },
]

Apply each check. For every row whose answer matches the old wording:

1. record the old answer, the circular and the reason in `flags` (an audit trail of every change);
2. replace the answer with the corrected one;
3. note the circular in a new column, `corrected_from_circular`.

`faq.index[matches]` gives the row labels where the pattern matched; `faq.loc[row, column]` reads or writes one cell.

In [ ]:
flags = []                                   # a record of every correction made
faq["corrected_from_circular"] = None

for rule in SUPERSEDED:
    # The rows whose answer matches the out-of-date pattern.
    matches = faq["answer"].str.contains(rule["pattern"], case=False, regex=True)

    for row in faq.index[matches]:
        flags.append({
            "faq_id": faq.loc[row, "faq_id"],
            "question": faq.loc[row, "question"],
            "old_answer": faq.loc[row, "answer"],
            "circular": rule["circular"],
            "reason": rule["reason"],
        })
        # Replace the answer, and note which circular justified it.
        faq.loc[row, "answer"] = rule["corrected"]
        faq.loc[row, "corrected_from_circular"] = rule["circular"]

pd.DataFrame(flags)

Notice the `(?<![\d.])` in the first pattern. Without it, "5%" also matches inside "2.5%", and the check would "correct"
the one answer that was already right. Write every check to be as specific as possible, then look at what it matched
before trusting it.

Each correction is recorded, with the circular that justifies it. That list goes to the web team too: the website is still
telling members the old rule.

**Your turn (1).** Circular FC/03/2025 changed life verification (the app is now allowed; home visits for pensioners aged
75+). Write a `SUPERSEDED` entry that would catch an answer saying verification must be done *in person at a branch*, and
check whether any published answer still says so.

In [ ]:
# Your code here

> **Where we are.** Every known superseded answer has been corrected, with an audit trail naming the circular. The
> remaining answers are believed correct. Next, find the rows that say the same thing.

## 5. Duplicates and paraphrases

Some rows are the same question worded differently, with the same answer. That's **good** training data: it teaches
the model that different wordings mean the same thing. But it creates a trap when you split into training and validation.
If one wording is in training and another in validation, validation is testing memory, not understanding. (The same trap
as Day 5's members with many payments.)

The fix is to give every group of paraphrases a **group id** and keep each group entirely on one side of the split.

How do you know two rows belong together? Here, by their answer: the same answer means the same question. Make a
simplified copy of each answer (`answer_key`: lower case, and `\W+`, meaning "any run of non-letter, non-digit
characters", replaced with a space) so tiny differences in punctuation don't matter. Then `ngroup()` gives every
distinct key its own number.

In [ ]:
# Exact duplicates (same question AND same answer) go.
faq = faq.drop_duplicates(subset=["question", "answer"])

# A simplified copy of each answer: lower case, punctuation removed. Same answer -> same key.
faq["answer_key"] = faq["answer"].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()

# ngroup() numbers the groups: every row with the same answer_key gets the same number.
faq["group_id"] = faq.groupby("answer_key").ngroup()

group_sizes = faq["group_id"].value_counts()      # how many rows in each group
print(len(faq), "rows in", faq["group_id"].nunique(), "answer groups;",
      (group_sizes > 1).sum(), "groups have paraphrases")

Look at three groups with more than one wording, to check the grouping makes sense:

In [ ]:
# Three of the groups that have more than one wording.
groups_with_paraphrases = group_sizes[group_sizes > 1].index[:3]
faq[faq["group_id"].isin(groups_with_paraphrases)].sort_values("group_id")[["group_id", "question"]]

> **Where we are.** Exact duplicates are gone and every paraphrase group has a group id, ready for a split that keeps
> groups together. The dataset so far teaches facts. It doesn't yet teach what to do when the right answer is "I can't".

### Theory: why behaviour examples matter so much

A model learns from what's in the data, and also from what **isn't**. If every training example is a question followed
by a confident answer, the model learns that *every question gets a confident answer*, including "What is my contribution
balance?", which it can't possibly know.

Behaviour examples show the pattern for the hard cases: the question, then the right **non-answer** (*"I can't see
personal records. You can check your balance in the app or at any branch."*). A few dozen of them shape the model's
behaviour far more than their number suggests, because they're the only examples of that pattern.

This is also where fine-tuning is strongest (see the table at the top): it's much better at teaching *how to behave*
than at teaching facts.

## 6. Teach behaviour, not just facts

The FAQ tells the model *what* to say. It doesn't show the model *what not to do*. Add examples of the behaviour you want
when the right answer is "I can't", because a member-service assistant will be asked those things every day:

- questions that need personal records it can't see;
- requests to change sensitive details (never by chat);
- questions outside the Fund's services, or asking for personal advice;
- possible fraud (an "agent" asking for money).

Each is a (question, answer) pair written by trainers.

In [ ]:
BEHAVIOUR = [
    ("What is my contribution balance?",
     "I can't see personal records. You can check your balance and statement on the member portal, the mobile app, the USSD "
     "service, or at any branch with your NIDA card."),
    ("Can you tell me my member number? My NIDA is 19850312-12345-67890-12.",
     "I can't look up personal records, and please don't share your NIDA number in a chat. Visit a branch with your NIDA card, "
     "or use 'Forgot member number' on the member portal."),
    ("Can you change my bank account to a new one?",
     "No. For your protection, bank details can only be changed at a branch or through the member portal after identity "
     "verification. The Fund never changes bank details by chat, phone, SMS or email."),
    ("Who will win the election?",
     "I can only help with questions about the Fund's services and benefits."),
    ("Should I invest my lump sum in land or in shares?",
     "I can't give personal investment advice. I can explain how your benefits are calculated and paid. For investment "
     "decisions, please speak to a licensed financial adviser."),
    ("Nisaidie kujua salio la michango yangu.",
     "Siwezi kuona taarifa zako binafsi. Unaweza kuangalia salio na taarifa ya michango kupitia mfumo wa wanachama, programu "
     "ya simu, huduma ya USSD, au tawi lolote ukiwa na kitambulisho cha NIDA."),
    ("An agent says he can speed up my claim for 50,000 shillings. Should I pay him?",
     "No. Do not pay anyone to process a claim. Benefits are paid only to you or your verified dependant, and the Fund never "
     "uses agents. Please report the offer to the Fund's fraud hotline."),
]

Turn the behaviour examples into a table with the same columns as the FAQ, so the two can be stacked:

- `category` and `source` say where each example came from (useful for the data card and for audits);
- the language is English, except the one Swahili example;
- each behaviour example gets its own new group number, continuing after the FAQ's numbers, so it never shares a group
  with an FAQ row.

`pd.concat` stacks the two tables; `ignore_index=True` renumbers the rows.

In [ ]:
behaviour = pd.DataFrame(BEHAVIOUR, columns=["question", "answer"])
behaviour["category"] = "behaviour"
behaviour["source"] = "written by trainers"

# All English except the one Swahili example.
behaviour["language"] = "en"
behaviour.loc[behaviour["question"].str.contains("Nisaidie"), "language"] = "sw"

# Each behaviour example is its own group, numbered after the FAQ groups.
first_new_group = faq["group_id"].max() + 1
behaviour["group_id"] = range(first_new_group, first_new_group + len(behaviour))

# Stack the FAQ and the behaviour examples into one table.
faq["source"] = "faq_export"
data = pd.concat([faq, behaviour], ignore_index=True)
print(len(data), "examples")

**Your turn (2).** Add three behaviour examples of your own: situations you've seen members raise that an assistant must
handle carefully. For each, write the answer you'd want a new colleague to give.

In [ ]:
# Your code here

### Optional: drafting more examples with an LLM, and why a person must review them

A local LLM (large language model) can draft extra question-answer pairs from handbook sections. It's a genuine
time-saver, but everything it drafts is a *draft*: it can misread a rule or invent one. The cells below draft a few
from one section and mark them `needs_review`. They're **not** added to the training set until a person has checked
each one against the handbook.

The schema (as in Day 7 Lab 2) forces the reply into a list called `pairs`, where each item has a `question` and an
`answer`.

In [ ]:
section = Path("handbook", "10_unemployment_benefit.md").read_text(encoding="utf-8")

# Ask for a list of {question, answer} pairs, in a fixed JSON shape (as in Day 7 Lab 2).
PAIRS_SCHEMA = {
    "type": "object",
    "properties": {
        "pairs": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"question": {"type": "string"}, "answer": {"type": "string"}},
                "required": ["question", "answer"],
            },
        },
    },
    "required": ["pairs"],
}

prompt = ("From this handbook section, write 3 questions a member might ask, each with an answer taken ONLY from the "
          "section.\n\n" + section)

Send the request to Ollama. If Ollama isn't running (or anything else goes wrong), `except Exception` catches the error
and prints a short message instead of stopping the notebook, because this part is optional. `display` shows the table
nicely inside the notebook.

In [ ]:
try:
    body = {
        "model": "qwen2.5:1.5b",
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": PAIRS_SCHEMA,
        "options": {"temperature": 0},
    }
    reply = requests.post("http://localhost:11434/api/chat", json=body, timeout=300).json()
    pairs = json.loads(reply["message"]["content"])["pairs"]

    # Every draft is marked as needing a person's review.
    drafts = pd.DataFrame(pairs)
    drafts["source"] = "LLM draft from 10_unemployment_benefit.md"
    drafts["status"] = "needs_review"
    display(drafts)
except Exception as error:
    print("Skipped (Ollama not available):", type(error).__name__)

Check each draft against `handbook/10_unemployment_benefit.md` and note any mistakes. In a real project you'd keep a
review log: who checked each pair, when, and what they changed.

> **Where we are.** The dataset now holds cleaned FAQ answers and behaviour examples, each with a category, a source and
> a group. LLM drafts, if you made any, are kept aside until a person reviews them. Now put every example in the shape
> the model will be trained on.

## 7. The chat format

Instruction-tuned models are trained on conversations with roles (the same `system`, `user` and `assistant` roles you used
on Day 7). Every example gets the **same system prompt**: the standing instruction you'll also use when the model is
deployed. Training and use must match: a model trained with one system prompt and deployed with another behaves less
predictably.

`to_chat` turns one question-answer pair into a conversation of three messages. Printing one with `json.dumps(...,
indent=2)` shows exactly what a training example looks like.

In [ ]:
SYSTEM = ("You are the member-service assistant of a social security fund in Tanzania. Answer briefly and accurately from "
          "the Fund's rules. If a question needs personal records, is outside the Fund's services, or asks for personal "
          "advice, say so politely and point to the right channel. Reply in the language of the question.")


def to_chat(question, answer):
    """One question-answer pair as a conversation: system prompt, member's question, assistant's answer."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    }


first = data.iloc[0]
print(json.dumps(to_chat(first["question"], first["answer"]), indent=2, ensure_ascii=False))

> **Where we are.** Every example is a three-message conversation with the same system prompt the deployed model will
> use. Before saving, make sure the tests in Lab 2 and Lab 3 will be honest.

### Theory: three sets, three jobs

Fine-tuning uses three separate sets of questions, each with its own job:

| Set | Job | Rule |
|---|---|---|
| **training** | what the model learns from | can include paraphrases of each other |
| **validation** | checked during training, to spot overfitting (Lab 2) | no paraphrase group shared with training |
| **held-out** | the final exam, written by staff in their own words (Lab 3) | never trained on, not even a close paraphrase |

If a question, or a close rewording of it, appears in both training and a test set, the test measures **memory**, not
whether the model learned to answer the Fund's questions. That's leakage again, in the form you met on Day 5: many rows
for the same underlying thing, spread across the split.

## 8. Split without leakage, and check against the held-out questions

Split the examples into **training** (what the model learns from) and **validation** (what you check it on during
training). `GroupShuffleSplit` (Day 5) keeps every answer group entirely on one side, so paraphrases never cross the
split. Behaviour examples always go to training: they're the patterns the model must learn, and there are only a few.

The check at the end counts groups found on both sides: it must be 0.

In [ ]:
faq_part = data[data["source"] == "faq_export"]
behaviour_part = data[data["source"] != "faq_export"]

# 15% of the answer groups for validation, the rest for training.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_rows, val_rows = next(splitter.split(faq_part, groups=faq_part["group_id"]))

train = pd.concat([faq_part.iloc[train_rows], behaviour_part])     # behaviour examples always train
val = faq_part.iloc[val_rows]

shared_groups = set(train["group_id"]) & set(val["group_id"])
print("train:", len(train), "examples | validation:", len(val), "examples")
print("answer groups on both sides:", len(shared_groups))

`heldout_questions.csv` is the final exam for Lab 3. None of its questions (or close paraphrases of them) may appear in
the training data, or Lab 3 will measure memory. Check with a similarity search: turn every question into a TF-IDF vector
(Day 6) and measure how similar each held-out question is to its closest training question.

**Cosine similarity** compares two vectors by the angle between them: 1 means the questions use the same words in the
same proportions, 0 means they have no words in common. The result `sim` is a table with one row per held-out question
and one column per training question.

In [ ]:
heldout = pd.read_csv("heldout_questions.csv")

# Learn one vocabulary from both sets, then turn each set into vectors.
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
vectorizer.fit(pd.concat([train["question"], heldout["question"]]))
heldout_vectors = vectorizer.transform(heldout["question"])
train_vectors = vectorizer.transform(train["question"])

# One row per held-out question, one column per training question.
sim = cosine_similarity(heldout_vectors, train_vectors)
print(sim.shape)

For each held-out question, find its closest training question: `sim.argmax(axis=1)` gives the column (training
question) with the highest similarity in each row, and `sim.max(axis=1)` gives that similarity. Sort with the most
similar pairs at the top: those are the ones to inspect.

In [ ]:
# For each held-out question: its closest training question, and how similar they are.
closest = pd.DataFrame({
    "held-out question": heldout["question"],
    "closest training question": train["question"].values[sim.argmax(axis=1)],
    "similarity": sim.max(axis=1).round(2),
})
closest.sort_values("similarity", ascending=False).head(8)

High similarity isn't automatically a problem. *"What is the minimum monthly pension?"* in both sets is fine if the held-out
version is how staff naturally ask it. But anything above about 0.8 deserves a look. If it's essentially the same question,
remove it from training.

> **Where we are.** Training and validation share no paraphrase group, and you've checked which held-out questions are
> close to a training question. One more check before anything is saved.

## 9. A last check for personal data

Training data gets copied, shared and baked into model weights. Once a member's phone number is inside a model, there's
no reliable way to take it out again. Make sure no member's personal data is in it.

Each pattern describes one kind of identifier:

- **member number:** exactly 9 digits on their own (`\b` = word boundary);
- **phone number:** a Tanzanian mobile number, starting `+255`, `255` or `0`, then 6 or 7, then 8 more digits;
- **NIDA number:** the National ID format, 8-5-5-2 digits separated by dashes.

The loop checks every question and answer against each pattern and shows up to two hits.

In [ ]:
PII_PATTERNS = {
    "member number (9 digits)": r"\b\d{9}\b",
    "phone number": r"(?:\+?255|0)[67]\d{8}\b",
    "NIDA number": r"\b\d{8}-\d{5}-\d{5}-\d{2}\b",
}

# Every question and answer, in one column.
all_text = pd.concat([data["question"], data["answer"]])

for name, pattern in PII_PATTERNS.items():
    hits = all_text[all_text.str.contains(pattern, regex=True)]
    print(f"{name:26s} {len(hits)} hit(s)")
    for hit in hits.head(2):
        print("    ->", hit)

One hit: the behaviour example where a member pastes a NIDA (National Identification Authority) number into the chat.
It's a made-up number and a deliberate example, teaching the model to tell members *not* to share it. Real conversation
logs would need this check to come back empty, or the numbers masked, before they're used for training.

> **Where we are.** The dataset is clean, corrected, grouped, split without leakage, checked against the held-out exam
> and scanned for personal data. Save it, with the document that explains it.

## 10. Save the dataset and its data card

The training format is **JSON Lines** (`.jsonl`): one conversation per line, each line a JSON object. It's the format
most fine-tuning tools expect, and because each line stands alone, a file can be read one example at a time however
large it gets. `"w"` opens the file for writing (replacing any old version).

In [ ]:
with open(OUT / "train.jsonl", "w", encoding="utf-8") as f:
    for question, answer in zip(train["question"], train["answer"]):
        f.write(json.dumps(to_chat(question, answer), ensure_ascii=False) + "\n")     # one conversation per line

with open(OUT / "val.jsonl", "w", encoding="utf-8") as f:
    for question, answer in zip(val["question"], val["answer"]):
        f.write(json.dumps(to_chat(question, answer), ensure_ascii=False) + "\n")

print("saved", OUT / "train.jsonl", "and", OUT / "val.jsonl")

Finally the **data card**: a short document that travels with the dataset. It's written as an f-string (a text with
`{...}` placeholders filled in from your variables), so its numbers always match the data you actually saved.
`set(...)` removes repeated circular names; `sorted` puts them in order.

In [ ]:
# The circulars used for corrections, without repeats.
circulars_used = sorted(set(flag["circular"] for flag in flags))

card = f"""# Data card: member-service instruction dataset v1

- **Purpose:** fine-tune a small LLM to answer common member questions in the Fund's style, and to refuse or redirect
  questions it must not answer.
- **Sources:** website FAQ export ({len(faq)} cleaned rows); {len(behaviour)} behaviour examples written by trainers.
- **Removed:** drafts/tests, empty answers, archived answers.
- **Corrected:** {len(flags)} published answers superseded by circulars ({', '.join(circulars_used)}).
- **Languages:** {data['language'].value_counts().to_dict()}
- **Split:** {len(train)} train / {len(val)} validation, grouped by answer so paraphrases never cross the split.
- **Held-out test:** heldout_questions.csv ({len(heldout)} questions) checked for overlap; max similarity {sim.max():.2f}.
- **Personal data:** scanned for member numbers, phone numbers and NIDA numbers; one synthetic NIDA number kept deliberately.
- **Known gaps:** few Swahili examples; nothing on voluntary members' payment methods; rules are from the Training Edition
  handbook, not the Act.
- **Owner / review:** to be completed before any production use.
"""
(OUT / "data_card.md").write_text(card, encoding="utf-8")
print(card)

The data card is short, but it answers the questions an auditor, a successor or a governance committee will ask:
where the data came from, what was changed and why, and what it can't do. Day 10 comes back to it.

---

## Looking back, looking ahead

**What you did in this lab.** You turned a messy FAQ export into an instruction dataset you could sign off: cleaned HTML,
removed drafts, blanks and archived answers, corrected answers overtaken by circulars (with an audit trail), grouped
paraphrases, added behaviour examples, wrote every example in chat format with the deployment system prompt, split by
group, checked against the held-out questions, scanned for personal data, and saved the data with a data card.

**The ideas to carry forward.**

- The model will copy your mistakes; the data deserves most of the effort.
- "Published" isn't "correct"; check against the source of truth.
- Teach behaviour (refusals, redirections), not just facts.
- Split by group so paraphrases can't leak.
- Personal data in training data ends up inside the model.

**What happens next.** Lab 2 fine-tunes Qwen2.5-0.5B-Instruct on `data/train.jsonl` with LoRA, a method that trains a few
million new numbers instead of all the model's weights.

## Suggested answer to (1)

A rule for the old life-verification wording. Run it on the **original** export, archived rows included, so you can see
everything it would catch:

In [ ]:
verification_rule = {
    "pattern": r"verification must be done in person|verify only at a branch",
    "circular": "FC/03/2025",
    "reason": "App verification allowed from 1 April 2025; home visits for 75+",
}

# Check the ORIGINAL export (including archived rows), cleaned the same way.
original = pd.read_csv("faq_export.csv")
original["answer"] = original["answer_html"].map(clean_html)

matches = original["answer"].str.contains(verification_rule["pattern"], case=False, regex=True)
original.loc[matches, ["faq_id", "status", "answer"]]

Only the archived answer matches, and you've already removed it, so no published answer still carries the old rule. If
your first attempt was a loose pattern like `verification.*at a branch`, you'll have seen it also catch the bank-details
answers ("...at a branch or through the member portal after identity verification"). Too loose a check buries the real
problems in false alarms; too tight a check misses them. Test your patterns on the data before trusting them.